In [9]:
import pandas as pd
from io import BytesIO
from fpdf import FPDF
import requests
import pyodbc
from pathlib import Path

In [ ]:
conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=db_name;'
                      'Database=tb_name;'
                      'Trusted_Connection=yes;')

def get_query(COLECAO):
    return f"""
    SELECT
    *
    FROM
        TABELA T
    WHERE 
        T.COLECAO = '{COLECAO}' 
        """

query1 = get_query('INV26')

df_col1 = pd.read_sql_query(query1,conn)

MAPEMANETO FOTOS: ALTERAR O CAMINHO PARA MAPEAR AS FOTOS DA COLEÇÃO DESEJADA

In [ ]:
def obter_imagem(url_foto):
    try:
        # Faz a requisição para a URL
        response = requests.get(url_foto, timeout=60)
        # Transforma o conteúdo binário em um "arquivo de memória"
        image_data = BytesIO(response.content)
        return image_data
    except Exception as e:
        print(f"Erro ao baixar a imagem: {e}")
        return None


def gerar_catalogo(df):
    # criando página vazia
    pdf = FPDF(orientation='L', unit='mm', format='A4')
    pdf.add_page()

    # determinando parâmetros iniciais
    X_inicial = 10  # Posição horizontal inicial
    X_foto = 10  # Posição horizontal à ser incrementada
    Y_inicial = 15  # Posição vertical Inicial
    Y_foto = 15  # Posicação vertical à ser incrementada
    largura = 35  # Largura ocupada pela foto
    contador = 0  # Utilizado para pular linha e páginas no loop
    espacamento = 90  # Espaçamento vertical entre linhas
    cont_pg = 0
    
    # loop de fotos
    for row in df.itertuples():
        pdf.set_fill_color(246, 234, 214)
        pdf.set_font('Helvetica', 'B', 6)
        pdf.set_xy(X_foto, Y_foto - 5)  # 5mm acima da posição da foto
        pdf.cell(w=largura, h=4, text=str(row.LINHA), new_x="RIGHT", new_y="TOP", align='C', fill='True')
        
        # FOTOS
        # 2. SE a foto existir, coloca ela
        foto = obter_imagem(f'PATH_PICS/{row.PRODUTO}_{row.COR_PRODUTO}')

    # 2. SE a foto existir, coloca ela
        if foto is not None:
            try:
                pdf.image(foto, x=X_foto, y=Y_foto, w=largura)  
            except Exception:
                print(f"Erro ao renderizar imagem de {row.PRODCOR}")
                # Se der erro na imagem, podemos desenhar um retângulo vazio no lugar
                pdf.rect(X_foto, Y_foto, largura, largura + 20)
                pdf.set_y(Y_foto + largura - 11)  # Seta o Y logo abaixo da altura da foto
                pdf.set_x(X_foto)
                pdf.set_font('Helvetica', '', 7)
                pdf.cell(w=largura, h=5, text="Imagem Não Encontrada", new_x="RIGHT", new_y="TOP", align='C')
        else:
            # Se der erro na imagem, podemos desenhar um retângulo vazio no lugar
            pdf.rect(X_foto, Y_foto, largura, largura + 20)
            pdf.set_y(Y_foto + largura - 11)  # Seta o Y logo abaixo da altura da foto
            pdf.set_x(X_foto)
            pdf.set_font('Helvetica', '', 7)
            pdf.cell(w=largura, h=5, text="Imagem Não Encontrada", new_x="RIGHT", new_y="TOP", align='C')

        ## PRODUTO
        # Ele usa a mesma coordenada X da foto para ficar alinhado
        pdf.set_y(Y_foto + largura + 20)  # Seta o Y logo abaixo da altura da foto
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', 'B', 7)
        pdf.cell(w=largura, h=5, text=row.PRODCOR, new_x="RIGHT", new_y="TOP", align='C')
        
        # DESC_PRODUTO
        pdf.set_y(Y_foto + largura + 25)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=row.DESC_PRODUTO, new_x="RIGHT", new_y="TOP", align='c')
        
        # DESC_COR_PRODUTO
        pdf.set_y(Y_foto + largura + 28)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=row.DESC_COR_PRODUTO, new_x="RIGHT", new_y="TOP", align='c')
        
        # DATA_CAPSULA
        pdf.set_y(Y_foto + largura + 31)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=f'Entrada: {row.DATA_ENTRADA}', new_x="RIGHT", new_y="TOP", align='c')
        
        # DIAS_VENDA
        pdf.set_y(Y_foto + largura + 34)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=f'Dias Venda: {str(row.DIAS_VENDA)}', new_x="RIGHT", new_y="TOP", align='c')
        
        # RECEITA
        pdf.set_y(Y_foto + largura + 37)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=f'Semanal: {row.VALOR_LIQ} R$ | Total: {row.VALOR_LIQ_TOTAL} R$', new_x="RIGHT", new_y="TOP", align='c')
        #VO
        pdf.set_y(Y_foto + largura + 40)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=f'Preço: {row.VO}', new_x="RIGHT", new_y="TOP", align='c')
        #RANK
        pdf.set_y(Y_foto + largura + 43)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', 'B', 6)
        pdf.cell(w=largura, h=3, text=f'{row.ROW}º', new_x="RIGHT", new_y="TOP", align='c')

        contador = contador + 1
        cont_pg = cont_pg + 1
        if contador == 10 and cont_pg <50:
            pdf.add_page()
            X_foto, Y_foto = X_inicial, Y_inicial
            contador = 0
        elif contador == 5 and cont_pg <50:
            X_foto = X_inicial
            Y_foto = Y_foto + espacamento
        else:
            X_foto += 60
    caminho = Path(r"C:Imagens/")
    nome_arquivo = f"VENDA_SEMANAL {df['COLECAO_PRODUTO'].iloc[0]}.pdf"
    caminho_arquivo = caminho / nome_arquivo
    pdf.output(str(caminho_arquivo))

In [ ]:
gerar_catalogo(df_col1)